# NB99 FINAL — Validate and Package the Complete Reproducible Project

This final notebook validates the complete manuscript workflow from NB00 through NB13 and creates a single reproducibility ZIP for review and archiving.

**Excluded intentionally:** the raw dataset in `01_DATA/` and all private credentials/checkpoints.

**Included:** results, figures, tables, logs, exports, project documentation/configuration, and only the definitive scientific notebooks used in the final manuscript workflow. Superseded, archived, temporary, and housekeeping notebooks are not included.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, hashlib, shutil, platform, sys
from datetime import datetime, timezone
import pandas as pd

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATA = ROOT/'01_DATA'
NOTEBOOKS = ROOT/'02_NOTEBOOKS'
RESULTS = ROOT/'03_RESULTS'
MODELS = ROOT/'04_MODELS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'
EXPORTS = ROOT/'08_EXPORTS'
FINAL_ZIP = ROOT/'09_FINAL_ZIP'

for p in [NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    assert p.exists(), f'Missing project folder: {p}'

print('ROOT:', ROOT)


In [ ]:
# Definitive notebook set used in the final manuscript workflow.
# NB11 was a housekeeping cleanup utility and is intentionally excluded.
FINAL_NOTEBOOKS = [
    'NB00_Project_Setup.ipynb',
    'NB01_Data_Audit_EDA.ipynb',
    'NB02_Dimensionality_Reduction.ipynb',
    'NB03_Baseline_Models_FIXED.ipynb',
    'NB04_Hybrid_Models_FIXED.ipynb',
    'NB05_Deep_Tabular_Models_FIXED.ipynb',
    'NB06_Statistical_Comparison_Interpretability_FIXED.ipynb',
    'NB07_ROC_Learning_Curves.ipynb',
    'NB08_Extended_Baselines_Redundancy_Ablation.ipynb',
    'NB09_Statistical_Robustness_Calibration_Selective_R2.ipynb',
    'NB10_Publication_Quality_Figures_MDPI_FINAL_R13.ipynb',
    'NB12_Top2_Confusion_and_Learning_Curves_FIXED.ipynb',
    'NB13_Final_Publication_Figure_Polish.ipynb',
    'NB99_Package_All_Results_UPDATED.ipynb',
]

missing_notebooks = [n for n in FINAL_NOTEBOOKS if not (NOTEBOOKS/n).exists()]
assert not missing_notebooks, f'Missing definitive notebooks: {missing_notebooks}'
print('Definitive notebooks found:', len(FINAL_NOTEBOOKS))


In [ ]:
# Validate required outputs from the complete final workflow.
checks = []

def add_check(name, ok, detail):
    checks.append({'check': name, 'passed': bool(ok), 'detail': str(detail)})

required_dirs = [
    'NB00_SETUP','NB01_AUDIT_EDA','NB02_DIMENSIONALITY',
    'NB03_BASELINES','NB04_HYBRIDS','NB05_DEEP_TABULAR',
    'NB06_STATS','NB07_ROC_LEARNING','NB08_EXTENDED_ABLATION',
    'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE','NB12_TOP2_DIAGNOSTICS'
]
for name in required_dirs:
    p = RESULTS/name
    add_check(f'results_dir_{name}', p.exists() and any(p.iterdir()), p)

# Original benchmark stages.
row_checks = [
    (RESULTS/'NB03_BASELINES'/'baseline_metrics_by_fold.csv', 60, 'NB03_metric_rows_60'),
    (RESULTS/'NB03_BASELINES'/'baseline_oof_predictions.csv', 48000, 'NB03_oof_rows_48000'),
    (RESULTS/'NB04_HYBRIDS'/'hybrid_metrics_by_fold.csv', 90, 'NB04_metric_rows_90'),
    (RESULTS/'NB04_HYBRIDS'/'hybrid_oof_predictions.csv', 72000, 'NB04_oof_rows_72000'),
    (RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_metrics_by_fold.csv', 30, 'NB05_metric_rows_30'),
    (RESULTS/'NB05_DEEP_TABULAR'/'deep_tabular_oof_predictions.csv', 24000, 'NB05_oof_rows_24000'),
    (RESULTS/'NB06_STATS'/'all_model_metrics_by_fold.csv', 180, 'NB06_metric_rows_180'),
    (RESULTS/'NB06_STATS'/'all_oof_predictions.csv', 144000, 'NB06_oof_rows_144000'),
    (RESULTS/'NB07_ROC_LEARNING'/'learning_curve_by_fold.csv', 225, 'NB07_learning_curve_rows_225'),
    (TABLES/'NB07_ROC_LEARNING'/'roc_auc_by_model_seed.csv', 36, 'NB07_roc_rows_36'),
    (RESULTS/'NB08_EXTENDED_ABLATION'/'nb08_metrics_by_fold.csv', 300, 'NB08_metric_rows_300'),
    (RESULTS/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'all_metrics_harmonized.csv', 480, 'NB09_harmonized_metric_rows_480'),
    (RESULTS/'NB12_TOP2_DIAGNOSTICS'/'tabpfn_learning_curve_by_fold.csv', 75, 'NB12_tabpfn_learning_rows_75'),
    (RESULTS/'NB12_TOP2_DIAGNOSTICS'/'top2_learning_curve_summary.csv', 10, 'NB12_top2_summary_rows_10'),
]
for path, expected_rows, check_name in row_checks:
    ok = path.exists() and len(pd.read_csv(path)) == expected_rows
    add_check(check_name, ok, path)

# Final statistical outputs required by the manuscript.
final_stats = [
    TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'full11_model_ranking.csv',
    TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'manuscript_full11_calibration_table.csv',
    TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'manuscript_full11_selective_key_coverages.csv',
    TABLES/'NB08_EXTENDED_ABLATION'/'feature_ablation_summary.csv',
    TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'full11_corrected_resampled_tests_vs_reference.csv',
    TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'/'full11_equivalence_margin_sensitivity.csv',
]
for path in final_stats:
    add_check(f'final_stat_{path.name}', path.exists() and path.stat().st_size > 0, path)

# Final publication figures. NB13 produces the definitive versions of Figures 1, 4 and 8.
final_fig_dir = FIGURES/'FINAL_PUBLICATION'
final_figures = [
    'Figure1_Correlation_Matrix_PUBLICATION.svg',
    'Figure2_LDA_Projection_PUBLICATION.svg',
    'Figure3_FULL11_Benchmark_PUBLICATION.svg',
    'Figure4_Confusion_Matrices_TabPFN_LDAXGBoost_PUBLICATION.svg',
    'Figure5_ROC_LDAXGBoost_PUBLICATION.svg',
    'Figure6_Calibration_TabPFN_LDAXGBoost_PUBLICATION.svg',
    'Figure7_Accuracy_Coverage_PUBLICATION.svg',
    'Figure8_Learning_Curves_TabPFN_LDAXGBoost_PUBLICATION.svg',
]
for name in final_figures:
    path = final_fig_dir/name
    add_check(f'final_figure_{name}', path.exists() and path.stat().st_size > 0, path)

validation = pd.DataFrame(checks)
validation.to_csv(EXPORTS/'final_pipeline_validation.csv', index=False)

failed = validation[~validation.passed]
display(validation)
assert failed.empty, 'FINAL VALIDATION FAILED:\n' + failed.to_string(index=False)

print('All complete-workflow validation checks PASSED.')


In [ ]:
# Stage the final reproducibility package.
stage = ROOT/'_ZIP_STAGE_FINAL'
if stage.exists():
    shutil.rmtree(stage)
stage.mkdir(parents=True)

# Include generated outputs, but never the raw dataset.
for folder in [RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS]:
    if folder.exists():
        shutil.copytree(folder, stage/folder.name, dirs_exist_ok=True)

# Include only definitive scientific notebooks.
nb_stage = stage/'02_NOTEBOOKS'
nb_stage.mkdir()
for name in FINAL_NOTEBOOKS:
    shutil.copy2(NOTEBOOKS/name, nb_stage/name)

# Include project-level documentation/configuration when available.
for name in ['README_PROJECT.md','project_config.json']:
    src = ROOT/name
    if src.exists():
        shutil.copy2(src, stage/name)

package_info = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'project': 'DRY_BEAN_HYBRID_Q1',
    'dataset_included': False,
    'raw_images_included': False,
    'credentials_included': False,
    'raw_data_excluded_reason': 'Original morphometric observations are intentionally excluded from the public reproducibility package.',
    'definitive_notebooks': FINAL_NOTEBOOKS,
    'python_version': sys.version,
    'platform': platform.platform(),
    'validation_file': '08_EXPORTS/final_pipeline_validation.csv',
    'final_publication_figure_count': 8,
}
with open(stage/'PACKAGE_INFO.json','w',encoding='utf-8') as f:
    json.dump(package_info,f,indent=2,ensure_ascii=False)

# Create a SHA-256 manifest of every staged file except the manifest itself.
manifest_rows = []
for p in sorted(stage.rglob('*')):
    if p.is_file() and p.name != 'PACKAGE_MANIFEST.csv':
        sha = hashlib.sha256(p.read_bytes()).hexdigest()
        manifest_rows.append({
            'relative_path': str(p.relative_to(stage)),
            'size_bytes': p.stat().st_size,
            'sha256': sha
        })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(stage/'PACKAGE_MANIFEST.csv',index=False)

print('Staged files:', len(manifest))
display(manifest.tail(20))


In [ ]:
# Create/replace final ZIP.
zip_base = FINAL_ZIP/'DRY_BEAN_HYBRID_Q1_COMPLETE_REPRODUCIBLE'
zip_path = Path(str(zip_base)+'.zip')
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_base),'zip',root_dir=stage)
shutil.rmtree(stage)

zip_sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
zip_info = {
    'zip_path': str(zip_path),
    'size_bytes': zip_path.stat().st_size,
    'size_mb': round(zip_path.stat().st_size/1024/1024,2),
    'sha256': zip_sha,
}
with open(EXPORTS/'final_zip_info.json','w',encoding='utf-8') as f:
    json.dump(zip_info,f,indent=2)

print('FINAL ZIP created:', zip_path)
print('Size (MB):', zip_info['size_mb'])
print('SHA256:', zip_sha)
print('\nNB99 UPDATED completed successfully.')


After successful execution, use:

`09_FINAL_ZIP/DRY_BEAN_HYBRID_Q1_COMPLETE_REPRODUCIBLE.zip`

This package excludes the original dataset, raw images, private credentials, and superseded notebooks. It includes the complete final scientific workflow (NB00–NB10, NB12, NB13 and NB99), generated results, final publication figures, tables, logs, exports, validation records, and a SHA-256 manifest.
